In [ ]:
from pyspark.sql import SparkSession as ss

spark = ss.builder    .appName('calc')     .master("local[*]")     .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")     .config("spark.hadoop.fs.s3a.endpoint", "http://minio-storage:9000")     .config("spark.hadoop.fs.s3a.access.key", "minioadmin")     .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")     .config("spark.hadoop.fs.s3a.path.style.access", "true")     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")     .getOrCreate()


In [ ]:
import pyspark.sql.functions as sf

df = spark.read.parquet("s3a://test-bucket/silver/ecommerce_refined")
df.printSchema()
print("row count:", df.count())
df.groupBy('event_type').count().show()


In [ ]:
# 1. 이벤트 퍼널 (view -> cart -> purchase 전환율)
funnel_df = df.groupBy('event_type').count()
funnel = funnel_df.collect()
counts = {row['event_type']: row['count'] for row in funnel}

view_cnt = counts.get('view', 0)
cart_cnt = counts.get('cart', 0)
purchase_cnt = counts.get('purchase', 0)

print(f"view     : {view_cnt}")
print(f"cart     : {cart_cnt}" + (f"  (view->cart 전환율: {cart_cnt/view_cnt*100:.2f}%)" if view_cnt else ""))
print(f"purchase : {purchase_cnt}" + (f"  (cart->purchase 전환율: {purchase_cnt/cart_cnt*100:.2f}%)" if cart_cnt else ""))
print(f"전체 전환율 (view->purchase): {purchase_cnt/view_cnt*100:.2f}%" if view_cnt else "")

# 골드 레이어로 저장
funnel_df.write.mode('overwrite').parquet("s3a://test-bucket/gold/event_funnel")


In [ ]:
# 2. 카테고리 / 브랜드별 매출 Top 10 (purchase 이벤트 기준)
purchase_df = df.filter(sf.col('event_type') == 'purchase')

category_revenue = purchase_df.groupBy('category_code')     .agg(
        sf.sum('price').alias('revenue'),
        sf.count('*').alias('purchase_count')
    )     .orderBy(sf.desc('revenue'))

print("=== 카테고리별 매출 Top 10 ===")
category_revenue.show(10, truncate=False)

brand_revenue = purchase_df.groupBy('brand')     .agg(
        sf.sum('price').alias('revenue'),
        sf.count('*').alias('purchase_count')
    )     .orderBy(sf.desc('revenue'))

print("=== 브랜드별 매출 Top 10 ===")
brand_revenue.show(10, truncate=False)

# 골드 레이어로 저장
category_revenue.write.mode('overwrite').parquet("s3a://test-bucket/gold/category_revenue")
brand_revenue.write.mode('overwrite').parquet("s3a://test-bucket/gold/brand_revenue")


In [ ]:
# 3. 일별(event_date) 추이
daily_events = df.groupBy('event_date', 'event_type')     .count()     .orderBy('event_date', 'event_type')

print("=== 일별 이벤트 타입별 건수 ===")
daily_events.show(30, truncate=False)

daily_purchase = purchase_df.groupBy('event_date')     .agg(
        sf.count('*').alias('purchase_count'),
        sf.sum('price').alias('daily_revenue')
    )     .orderBy('event_date')

print("=== 일별 구매 건수 / 매출 ===")
daily_purchase.show(30, truncate=False)

# 골드 레이어로 저장
daily_events.write.mode('overwrite').parquet("s3a://test-bucket/gold/daily_events")
daily_purchase.write.mode('overwrite').parquet("s3a://test-bucket/gold/daily_purchase")


In [ ]:
# 4. 가격 분포 / 기술통계
price_stats_df = df.select('price').describe()
price_stats_df.show()

price_percentiles_df = df.select(
    sf.expr('percentile_approx(price, 0.25)').alias('p25'),
    sf.expr('percentile_approx(price, 0.5)').alias('median'),
    sf.expr('percentile_approx(price, 0.75)').alias('p75'),
    sf.expr('percentile_approx(price, 0.95)').alias('p95'),
)
price_percentiles_df.show()

# purchase 건만 따로 (실제 판매 가격대 분포)
print("=== purchase 이벤트만의 가격 분포 ===")
purchase_price_stats_df = purchase_df.select('price').describe()
purchase_price_stats_df.show()

# 골드 레이어로 저장
price_stats_df.write.mode('overwrite').parquet("s3a://test-bucket/gold/price_stats")
price_percentiles_df.write.mode('overwrite').parquet("s3a://test-bucket/gold/price_percentiles")
purchase_price_stats_df.write.mode('overwrite').parquet("s3a://test-bucket/gold/purchase_price_stats")
